<a href="https://colab.research.google.com/github/dbwofla11/LCK-Efficiency-Analysis/blob/basecode/%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%82%AC%EC%9D%B4%EC%96%B8%EC%8A%A4_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 오라클 엘릭서 부분


In [ ]:
# ============================================
# 1. Oracle's Elixir 데이터 가져오기
# ============================================

import pandas as pd

# 파일 읽기
df_pro = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv")

# 데이터 로드
file_path = "2025_LoL_esports_match_data_from_OraclesElixir.csv"

raw_df = pd.read_csv(file_path)

print(raw_df.shape)
print(raw_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '2025_LoL_esports_match_data_from_OraclesElixir.csv'

In [ ]:
print(raw_df.columns.tolist())

['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'inhibitors', '

In [ ]:
# 1. 유효한 포지션 리스트 정의
valid_positions = ['top', 'jng', 'mid', 'bot', 'sup']

# 2. 리스트에 포함된 행만 추출 (team이나 NaN은 자동으로 제외됨)
player_df = raw_df[raw_df['position'].isin(valid_positions)].copy()

# 결과 확인
print(f"필터링 후 데이터 크기: {player_df.shape}")
print(f"포지션 종류: {player_df['position'].unique()}")

필터링 후 데이터 크기: (100530, 165)
포지션 종류: ['top' 'jng' 'mid' 'bot' 'sup']


### 필요없는 경기 메타 컬럼 제거

In [ ]:
remove_cols = [
    'gameid',
    'datacompleteness',
    'url',
    'ban1', 'ban2', 'ban3', 'ban4', 'ban5',
    'pick1', 'pick2', 'pick3', 'pick4', 'pick5',
    'teamname',
    'playerid'
]

existing_remove_cols = [
    col for col in remove_cols
    if col in player_df.columns
]

player_df = player_df.drop(columns=existing_remove_cols)
player_df.columns

Index(['league', 'year', 'split', 'playoffs', 'date', 'game', 'patch',
       'participantid', 'side', 'position',
       ...
       'opp_csat25', 'golddiffat25', 'xpdiffat25', 'csdiffat25', 'killsat25',
       'assistsat25', 'deathsat25', 'opp_killsat25', 'opp_assistsat25',
       'opp_deathsat25'],
      dtype='object', length=150)

### 패치버전 정리
* 메이저 버전만 추출함
* 버그수정 버전 이런건 제외함

In [ ]:
player_df['patch_major'] = (
    player_df['patch']
    .astype(str)
    .str.extract(r'(\d+\.\d+)')
)

print(player_df['patch_major'].value_counts())

patch_major
15.09    8760
15.07    8200
15.08    7980
15.1     6520
15.15    6380
15.17    6230
15.16    6130
15.14    6020
15.03    5690
15.02    5360
15.06    4640
15.11    4460
15.13    3860
15.19    3770
15.04    3710
15.01    3150
15.05    2140
15.2     1920
15.18    1830
15.12    1680
15.24     750
15.21     600
15.23     560
15.22     190
Name: count, dtype: int64


In [ ]:
# 결측치 비율 확인
missing_ratio = (
    player_df.isnull().mean()
    .sort_values(ascending=False)
)

print(missing_ratio.head(30))

towers                    1.0
atakhans                  1.0
opp_atakhans              1.0
firstbaron                1.0
opp_void_grubs            1.0
void_grubs                1.0
opp_heralds               1.0
oceans                    1.0
chemtechs                 1.0
hextechs                  1.0
dragons (type unknown)    1.0
elders                    1.0
opp_elders                1.0
firstherald               1.0
heralds                   1.0
firstdragon               1.0
dragons                   1.0
opp_dragons               1.0
elementaldrakes           1.0
opp_elementaldrakes       1.0
infernals                 1.0
mountains                 1.0
clouds                    1.0
gpr                       1.0
opp_turretplates          1.0
turretplates              1.0
firsttothreetowers        1.0
firstmidtower             1.0
opp_towers                1.0
firsttower                1.0
dtype: float64


### 우리가 핵심적으로 쓸 특징의 결측치 제거

In [ ]:
core_features = [
    'golddiffat15',
    'xpdiffat15',
    'csdiffat15',
    'dpm',
    'earnedgold',
    'visionscore'
]

existing_core = [
    c for c in core_features
    if c in player_df.columns
]

player_df = player_df.dropna(subset=existing_core)
player_df.shape

(92360, 151)

# 일단 강건한 특징 추출을 위해 우리가 테스트해볼 칼럼

In [ ]:
selected_cols = [ # 여기 나와있는 거는 레퍼런스 자료 참고해서 일부만 뽑아온 것들임

    # 기본 정보
    'league',
    'patch_major',
    'date',
    'playername',
    'position',
    'champion',
    'result',

    # 라인전 강도
    'golddiffat15',
    'xpdiffat15',
    'csdiffat15',

    # 자원 및 성장
    'earnedgold',
    'earned gpm',
    'goldspent',

    # 전투 영향력
    'dpm',
    'damagetakenperminute',
    'damagemitigatedperminute',

    # 시야
    'visionscore',
    'vspm',
    'visionwardsbought',

    # 운영 참여
    'firstbloodkill',
    'firstbloodassist',

    #킬 참여
    'kills',
    'deaths',
    'assists',
    'teamkills',

    # 생존력
    'deaths',

    # CS 운영
    'cspm',

    # 오브젝트
    'monsterkills',
]

selected_cols = [
    c for c in selected_cols
    if c in player_df.columns
]

analysis_df = player_df[selected_cols].copy()

analysis_df.head()
analysis_df.columns

Index(['league', 'patch_major', 'date', 'playername', 'position', 'champion',
       'result', 'golddiffat15', 'xpdiffat15', 'csdiffat15', 'earnedgold',
       'earned gpm', 'goldspent', 'dpm', 'damagetakenperminute',
       'damagemitigatedperminute', 'visionscore', 'vspm', 'firstbloodkill',
       'firstbloodassist', 'kills', 'deaths', 'assists', 'teamkills', 'deaths',
       'cspm', 'monsterkills'],
      dtype='object')

# 요 아래부턴 하나씩 확인해보는거임 엘릭서 데이터만 이거 한다음에 Riot API랑 비교해볼 예정

In [ ]:
analysis_df = analysis_df.loc[:, ~analysis_df.columns.duplicated()]

In [ ]:
# 골드 효율성
analysis_df['dmg_per_gold'] = (
    analysis_df['dpm'] /
    (analysis_df['earnedgold'] + 1)
)

In [ ]:
# 저점 안정성
player_variance = (
    analysis_df
    .groupby('playername')['dpm']
    .std()
    .sort_values()
)

print(player_variance.head(20))

playername
Heart             1.042346
StoRm             6.148930
Termo             6.884179
Fla01             7.827248
BL Frechdachs     9.763377
Shiro            11.379894
PAPU Papulon     11.667050
Sawyor           13.930781
Druk             14.528782
Werlyb           14.535853
Erk              16.052738
Lizia            16.274487
Kartso           16.997591
Nainess          17.158582
Zeycce           18.164937
Kodie            18.694666
Basei            19.355987
Faith            19.845023
HST Terry        21.925543
PCS OQ07         22.259651
Name: dpm, dtype: float64


In [ ]:
# 라인전 일관성 ( 기복이 있는지 없는지 )
analysis_df['lane_score'] = (
    analysis_df['golddiffat15'] * 0.4 +
    analysis_df['xpdiffat15'] * 0.3 +
    analysis_df['csdiffat15'] * 0.3
)

In [ ]:
# 시야 효율 -> 시야를 땀에 있어서 효율적인가?
analysis_df['vision_efficiency'] = (
    analysis_df['visionscore'] /
    (analysis_df['deaths'] + 1)
)

In [ ]:
# 킬 비율
analysis_df['killparticipation'] = (
    (analysis_df['kills'] + analysis_df['assists']) /
    (analysis_df['teamkills'] + 1e-6)
)

In [ ]:
# 참여 효율
# 필요한 교전만 하는가??
analysis_df['kp_per_death'] = (
    analysis_df['killparticipation'] /
    (analysis_df['deaths'] + 1)
)

In [ ]:
# 패치별 분포 확인
# 패치가 달라도 분포가 일정하고 안정적인지 확인해봄

patch_summary = (
    analysis_df
    .groupby('patch_major')[
        ['dmg_per_gold', 'lane_score']
    ]
    .mean()
)

print(patch_summary)

             dmg_per_gold    lane_score
patch_major                            
15.01            0.070031  4.179663e-17
15.02            0.071227  2.778497e-16
15.03            0.070189 -8.991050e-17
15.04            0.072317 -1.900540e-16
15.05            0.071313 -2.656235e-16
15.06            0.072261 -9.849107e-17
15.07            0.072246  2.142219e-16
15.08            0.073185  1.023496e-16
15.09            0.070925 -1.100195e-17
15.1             0.069381 -8.923614e-17
15.11            0.070417  2.915047e-17
15.12            0.074270 -1.691768e-17
15.13            0.072785 -4.049725e-16
15.14            0.071067  5.133979e-17
15.15            0.071460  1.224634e-16
15.16            0.071933  3.565633e-16
15.17            0.071471  3.803267e-16
15.18            0.076365  4.814607e-16
15.19            0.076350 -3.015566e-17
15.2             0.073215  2.442491e-16
15.21            0.080781 -8.289665e-16
15.22            0.076615 -1.121910e-15
15.23            0.080927 -4.060244e-16


In [ ]:
league_summary = (
    analysis_df
    .groupby('league')[
        ['dmg_per_gold', 'lane_score']
    ]
    .mean()
)

print(league_summary)

             dmg_per_gold    lane_score
league                                 
AL               0.074387  5.917148e-16
ASI              0.079463  8.628019e-16
Asia Master      0.079408 -4.421155e-16
CD               0.069563 -1.239319e-16
CT               0.067570 -9.094947e-16
DCup             0.063644  2.273737e-16
EBL              0.078572  2.380876e-16
EM               0.070087  9.043271e-17
EWC              0.071947 -5.598215e-16
FST              0.071485 -3.248195e-16
HC               0.077990  4.476419e-16
HLL              0.071934  5.015596e-16
HM               0.073985 -6.881045e-16
HW               0.075163 -5.983518e-16
IC               0.077565 -7.612958e-16
KeSPA            0.080927 -4.060244e-16
LAS              0.081092  3.632632e-16
LCK              0.068536  1.638729e-16
LCKC             0.070059 -7.341401e-17
LCP              0.067136  3.906764e-17
LEC              0.063750  1.486102e-16
LFL              0.065008  3.932564e-16
LFL2             0.068589 -3.842420e-16


# csv로 내보내기 전 데이터 구조 확인

In [ ]:
analysis_df.head()

,league,patch_major,date,playername,position,champion,result,golddiffat15,xpdiffat15,csdiffat15,...,deaths,assists,teamkills,cspm,monsterkills,dmg_per_gold,lane_score,vision_efficiency,killparticipation,kp_per_death
0,LFL2,15.01,2025-01-11 11:11:24,PatkicaA,top,Gnar,0,-841.0,-191.0,-6.0,...,2,1,3,8.8191,0,0.106304,-395.5,5.666667,0.666666,0.222222
1,LFL2,15.01,2025-01-11 11:11:24,Joinze,jng,Maokai,0,-828.0,-16.0,6.0,...,3,1,3,5.3894,132,0.047875,-334.2,7.250000,0.333333,0.083333
2,LFL2,15.01,2025-01-11 11:11:24,Sayn,mid,Hwei,0,-1085.0,-501.0,-25.0,...,2,0,3,7.8769,0,0.095432,-591.8,6.666667,0.333333,0.111111
3,LFL2,15.01,2025-01-11 11:11:24,Shiganari,bot,Jinx,0,-911.0,-200.0,-5.0,...,3,1,3,9.4598,12,0.044176,-425.9,2.500000,0.666666,0.166667
4,LFL2,15.01,2025-01-11 11:11:24,Lekcyc,sup,Leona,0,-172.0,439.0,14.0,...,3,2,3,1.4322,0,0.071603,67.1,20.500000,0.666666,0.166667


In [ ]:
analysis_df.columns

Index(['league', 'patch_major', 'date', 'playername', 'position', 'champion',
       'result', 'golddiffat15', 'xpdiffat15', 'csdiffat15', 'earnedgold',
       'earned gpm', 'goldspent', 'dpm', 'damagetakenperminute',
       'damagemitigatedperminute', 'visionscore', 'vspm', 'firstbloodkill',
       'firstbloodassist', 'kills', 'deaths', 'assists', 'teamkills', 'cspm',
       'monsterkills', 'dmg_per_gold', 'lane_score', 'vision_efficiency',
       'killparticipation', 'kp_per_death'],
      dtype='object')

In [ ]:
analysis_df.to_csv(
    'oracle_elixir_processed.csv',
    index=False
)

# RIOT API불러오는 부분


In [ ]:
import requests
import pandas as pd

# Riot Developer Portal에서 발급
API_KEY = "RGAPI-524ed0b3-813e-47e8-b51c-46aa458fbdee"

headers = {
    "X-Riot-Token": API_KEY
}

In [ ]:
# 예시 소환사명
summoner_name = "Hide on bush"
tag_line = "KR1"

# Riot ID -> PUUID
url = f"https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{summoner_name}/{tag_line}"

response = requests.get(url, headers=headers)

account_data = response.json()

print(account_data)

puuid = account_data['puuid']
print("PUUID:", puuid)

{'puuid': 'lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg', 'gameName': 'Hide on bush', 'tagLine': 'KR1'}
PUUID: lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg


In [ ]:
rank_qu = "https://kr.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
response = requests.get(rank_qu, headers=headers)

challenger_data = response.json()

challenger_data = pd.DataFrame(challenger_data)
challenger_data.head()
print(challenger_data.shape)
print(challenger_data.columns)


(300, 5)
Index(['tier', 'leagueId', 'queue', 'name', 'entries'], dtype='object')


In [ ]:
challenger_data

,tier,leagueId,queue,name,entries
0,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '6HyoDnID4EhYPu6cZW_LLNBWtZ4IyhVsWFs...
1,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'CMqC1inIb3bFqPqIuX0c5gf6UV6OCCVSS-F...
2,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'zPvegHlhyCZqERlm_txkPyIlhQvH8brnvRQ...
3,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'Q_5rN7n8B0-8OtKj_hJtpJhrjiPiEt9Juxt...
4,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'xKWFpdw63ALi0IPwhO9WEcWyM-_Aw0fu_S6...
...,...,...,...,...,...
295,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'B-qJtm1WMI-6334Zm5OKLUr71GRtWczTlpk...
296,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '4uqbKHfx1c3PoiWk6YcmJm6WHpdOjwuoZ03...
297,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'G_YoCMVVT0qFZJOCLyAjkzDE5yrgxmH12h9...
298,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '9xZG4d930EkVGAxNLR8ceaQgvyaYhamnYg8...


### 챌린저 유저들 정보 자세히 보기

In [ ]:
challenger_players = pd.json_normalize(
    challenger_data['entries']
)

challenger_players.head()

# | 컬럼           | 의미         |
# | ------------ | ---------- |
# | puuid        | 플레이어 고유 ID |
# | leaguePoints | LP         |
# | wins/losses  | 승률         |
# | veteran      | 장기 챌 여부    |
# | hotStreak    | 연승 여부      |


,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak
0,6HyoDnID4EhYPu6cZW_LLNBWtZ4IyhVsWFsxsmfHvRapkO...,2591,I,325,270,True,False,False,False
1,CMqC1inIb3bFqPqIuX0c5gf6UV6OCCVSS-FUZxiwy9rfjj...,2552,I,332,285,True,False,False,False
2,zPvegHlhyCZqERlm_txkPyIlhQvH8brnvRQnS6qAeElZTV...,2535,I,194,139,True,False,False,False
3,Q_5rN7n8B0-8OtKj_hJtpJhrjiPiEt9Juxty2f7aYA9PiP...,2524,I,232,173,True,False,False,True
4,xKWFpdw63ALi0IPwhO9WEcWyM-_Aw0fu_S6e6_MNE1iHZE...,2514,I,231,172,True,False,False,False


# 유저 정보 안에서 매치 아이디 수집 후 메치 데이터 가져오기

-> 일단 어캐 하는지 모르니까 한 사람 데이터만 가져와서 테스트 해봄

In [ ]:
import requests
import time

API_KEY = "RGAPI-524ed0b3-813e-47e8-b51c-46aa458fbdee"

headers = {
    "X-Riot-Token": API_KEY
}

def get_match_ids(puuid, count=20):

    url = (
        f"https://asia.api.riotgames.com/"
        f"lol/match/v5/matches/by-puuid/"
        f"{puuid}/ids?start=0&count={count}"
    )

    response = requests.get(
        url,
        headers=headers
    )

    print("MATCH IDS:", response.status_code)

    if response.status_code != 200:
        return []

    return response.json()



sample_puuid = challenger_players.iloc[0]['puuid'] # 챌린저 한명 데이터 뽑기

match_ids = get_match_ids(sample_puuid)

print(match_ids)
len(match_ids)



MATCH IDS: 200
['KR_8205660163', 'KR_8205646533', 'KR_8205625459', 'KR_8205603025', 'KR_8205563308', 'KR_8205521328', 'KR_8204871684', 'KR_8204469050', 'KR_8204440346', 'KR_8204410464', 'KR_8204370954', 'KR_8202790500', 'KR_8202779253', 'KR_8202766390', 'KR_8202746845', 'KR_8199299071', 'KR_8199270099', 'KR_8199156929', 'KR_8199128164', 'KR_8199064553']


20

In [ ]:
def get_match_data(match_id):

    url = (
        f"https://asia.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}"
    )

    response = requests.get(
        url,
        headers=headers
    )

    print("MATCH:", response.status_code)

    if response.status_code != 200:
        return None

    return response.json()

match_data = get_match_data(match_ids[0])

print(match_data.keys())

MATCH: 200
dict_keys(['metadata', 'info'])


In [ ]:
def extract_features(
    match_data,
    puuid,
    summoner_name=None,
    tier='CHALLENGER'
):

    participants = match_data['info']['participants']

    player = None

    for p in participants:
        if p['puuid'] == puuid:
            player = p
            break

    if player is None:
        return None

    game_minutes = (
        match_data['info']['gameDuration'] / 60
    )

    # 팀 총 킬
    team_kills = sum([
        x['kills']
        for x in participants
        if x['teamId'] == player['teamId']
    ])

    features = {

        # 🔥 유저 정보
        'puuid': puuid,
        'summonerName': summoner_name,
        'tier': tier,

        # 기본 정보
        'champion': player['championName'],
        'position': player['teamPosition'],

        # 전투
        'kills': player['kills'],
        'deaths': player['deaths'],
        'assists': player['assists'],

        # 효율
        'dpm':
            player['totalDamageDealtToChampions']
            / game_minutes,

        'cspm':
            (
                player['totalMinionsKilled']
                + player['neutralMinionsKilled']
            ) / game_minutes,

        'visionscore':
            player['visionScore'],

        'earnedgold':
            player['goldEarned'],

        'killparticipation':
            (
                player['kills']
                + player['assists']
            ) / max(team_kills, 1),

        'dmg_per_gold':
            (
                player['totalDamageDealtToChampions']
                / max(player['goldEarned'], 1)
            ),

        'vision_efficiency':
            (
                player['visionScore']
                / (player['deaths'] + 1)
            )
    }

    return features

In [ ]:
features = extract_features(
    match_data,
    sample_puuid
)

print(features)

{'puuid': '6HyoDnID4EhYPu6cZW_LLNBWtZ4IyhVsWFsxsmfHvRapkOgQkS0S7aoxNdGUlvOsFimjGgU2Qe_FbQ', 'summonerName': None, 'tier': 'CHALLENGER', 'champion': 'Ryze', 'position': 'MIDDLE', 'kills': 4, 'deaths': 3, 'assists': 3, 'dpm': 942.5818882466282, 'cspm': 8.709055876685936, 'visionscore': 20, 'earnedgold': 10442, 'killparticipation': 0.5, 'dmg_per_gold': 2.342463129668646, 'vision_efficiency': 5.0}


In [ ]:
challenger_players.shape

(300, 9)

# 여러 게임 데이터 수집하기

In [ ]:
all_features = []

for idx, row in challenger_players.iterrows():

    puuid = row['puuid']

    summoner_name = row.get(
        'summonerName',
        'Unknown'
    )

    try:

        match_ids = get_match_ids(
            puuid,
            count=10
        )

        for match_id in match_ids:

            match_data = get_match_data(match_id)
            time.sleep(1.2)

            features = extract_features(
                match_data,
                puuid,
                summoner_name=summoner_name,
                tier='CHALLENGER'
            )

        if features:
            all_features.append(features)
        else:
            print("Feature extraction failed")

            time.sleep(2000)

    except Exception as e:
        print(e)

MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH IDS: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200
MATCH: 200


In [ ]:
soloq_df = pd.DataFrame(all_features)

soloq_df

,puuid,summonerName,tier,champion,position,kills,deaths,assists,dpm,cspm,visionscore,earnedgold,killparticipation,dmg_per_gold,vision_efficiency
0,6HyoDnID4EhYPu6cZW_LLNBWtZ4IyhVsWFsxsmfHvRapkO...,Unknown,CHALLENGER,Cassiopeia,MIDDLE,7,8,6,1106.287839,8.475836,39,13112,0.406250,2.647880,4.333333
1,CMqC1inIb3bFqPqIuX0c5gf6UV6OCCVSS-FUZxiwy9rfjj...,Unknown,CHALLENGER,Sylas,MIDDLE,6,5,4,855.721649,9.226804,24,8975,0.322581,1.849694,4.000000
2,zPvegHlhyCZqERlm_txkPyIlhQvH8brnvRQnS6qAeElZTV...,Unknown,CHALLENGER,Nidalee,JUNGLE,7,0,9,649.444967,8.353716,26,9582,0.695652,1.200793,26.000000
3,Q_5rN7n8B0-8OtKj_hJtpJhrjiPiEt9Juxty2f7aYA9PiP...,Unknown,CHALLENGER,Khazix,,20,2,2,3660.249827,0.000000,0,16761,0.309859,5.244735,0.000000
4,xKWFpdw63ALi0IPwhO9WEcWyM-_Aw0fu_S6e6_MNE1iHZE...,Unknown,CHALLENGER,Rengar,JUNGLE,11,1,2,705.081081,9.135135,41,10507,0.541667,1.241458,20.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,B-qJtm1WMI-6334Zm5OKLUr71GRtWczTlpkydbWSjDX7Az...,Unknown,CHALLENGER,Jhin,BOTTOM,8,4,12,889.035040,7.795148,35,16569,0.555556,1.658881,7.000000
296,4uqbKHfx1c3PoiWk6YcmJm6WHpdOjwuoZ03hu08CqVnhCG...,Unknown,CHALLENGER,Thresh,UTILITY,4,5,26,388.611670,1.569416,70,9543,0.652174,1.011946,11.666667
297,G_YoCMVVT0qFZJOCLyAjkzDE5yrgxmH12h9nocpcZmO43D...,Unknown,CHALLENGER,Caitlyn,BOTTOM,9,7,14,1582.514970,10.059880,38,18790,0.621622,2.812986,4.750000
298,9xZG4d930EkVGAxNLR8ceaQgvyaYhamnYg8b2QUnTEaYGx...,Unknown,CHALLENGER,Elise,UTILITY,5,12,21,562.401639,0.639344,157,12029,0.481481,1.901322,12.076923


In [ ]:
soloq_df.shape
print(soloq_df["puuid"].value_counts())

In [ ]:
player_mean = (
    soloq_df
    .groupby('puuid')[
        ['dpm', 'cspm']
    ]
    .mean()
)

print(player_mean)

In [ ]:
player_std = (
    soloq_df
    .groupby('puuid')['dpm']
    .std()
)

print(player_std)